In [ ]:
from dataclasses import replace

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathtracer import Renderer, lattice_preview

from stats import principal_components, independent_components, sparse_coding
from spectral import lattice_geometry, row_twiddle, fft, ifft, bin_freq, alias_radius_sq

---
# DATA
Renders the test scene once: a colour image for PCA and a brightness image for every other section.

- `beauty` and `luminance` are scene-referred linear values in cd/m^2, not display codes
- `rendercam` is the single source of camera state: later sections read their lattice geometry from it
- `show_sky` makes primary misses return environment radiance, so every pixel sees light rather than a black void
- Change the camera, sky or AOVs here and rerun all; only STATISTICS renders again, its second half under the same settings

In [ ]:
scene = Renderer("scenes/cornell.json")
rendercam = replace(scene.default_camera)
show_sky = True
frame = scene.render(camera=rendercam, aovs=("beauty", "luminance"), show_sky=show_sky)

---
# SPECTRAL
Splits the brightness image into waves of different sizes and directions: how much each contributes, and how much is render noise.

- Lattice-aware DFT: a per-row twiddle moves odd rows to their true positions, so square and hex share one path
- Frequencies are cycles per column pitch; each bin is folded to its nearest reciprocal-lattice point for its true radius and angle
- Radial power excludes DC and stops at the lattice Nyquist; the spatial frequency curve is f·P(f), so its peak is the dominant scale
- Phase is referenced to the luminance centroid, so it does not depend on where the scene sits in frame
- Log magnitude is dB re the peak bin; zero bins are NaN and left out of the noise floor histogram

In [ ]:
row_pitch, row_shift = lattice_geometry(rendercam)
img = frame["luminance"][..., 0]
twiddle = row_twiddle(img.shape, row_shift)
spectrum = fft(img, twiddle)
fy, fx = bin_freq(img.shape, row_pitch)
radius_sq, true_fy, true_fx = alias_radius_sq(fy, fx, row_pitch, row_shift)
radius = np.sqrt(radius_sq)
mag = np.abs(spectrum)

bin_width = 1.0 / min(img.shape[1], img.shape[0] * row_pitch)
nyquist = 0.5 * min(1.0 / row_pitch, np.hypot(1.0, row_shift / row_pitch))
# DC excluded
power, edges = np.histogram(radius, bins=np.arange(0.0, radius.max() + bin_width, bin_width), weights=np.where(radius > 0, mag**2, 0.0))
centres = 0.5 * (edges[1:] + edges[:-1])
inside = edges[1:] <= nyquist
log_power = centres * power
peak = np.argmax(np.where(inside, log_power, 0.0))
print(f"Frequency at which power per log frequency is greatest: {centres[peak]:.4f} cycles/pitch")

total = img.sum()
cx = (np.arange(img.shape[1]) @ img.sum(axis=0) + row_shift * img[1::2].sum()) / total
cy = row_pitch * (np.arange(img.shape[0]) @ img.sum(axis=1)) / total
phase = np.angle(spectrum * np.exp(2j * np.pi * (fx[None, :] * cx + fy[:, None] * cy)))
log_mag = 20 * np.log10(mag / mag.max(), out=np.full_like(mag, np.nan), where=mag > 0)

In [ ]:
fig, (ax_lum, ax_mag, ax_phase) = plt.subplots(1, 3, figsize=[30,10])
ax_lum.imshow(img)
ax_lum.set_title("Luminance")

ax_mag.imshow(np.fft.fftshift(log_mag), cmap="binary")
ax_mag.set_title("Log magnitude")

ax_phase.imshow(np.fft.fftshift(phase), cmap="binary")
ax_phase.set_title("Phase")

In [ ]:
power_2d = mag**2
theta = np.arctan2(true_fy, true_fx)

fig, ax = plt.subplots(1, 5, figsize=[25, 5])

ax[0].plot(centres[inside], log_power[inside], lw=1)
ax[0].axvline(centres[peak], color="r", lw=1)
ax[0].set_title("Spatial frequency")
ax[0].set_xlabel("Radius (cycles/pitch)"), ax[0].set_ylabel("Power per log frequency")
ax[0].set_ylim(bottom=0)

ax[1].plot(edges[1:], np.cumsum(power) / power.sum(), lw=1)
ax[1].set_title("Cumulative energy")
ax[1].set_xlabel("Radius (cycles/pitch)"), ax[1].set_ylabel("Fraction of AC power")

ax[2].semilogy(np.fft.fftshift(fx), np.fft.fftshift(power_2d.sum(axis=0)), lw=1, label="fx")
ax[2].semilogy(np.fft.fftshift(fy), np.fft.fftshift(power_2d.sum(axis=1)), lw=1, label="fy")
ax[2].set_title("Marginal spectra")
ax[2].set_xlabel("Frequency (cycles/pitch)"), ax[2].set_ylabel("Power")
ax[2].legend()

ax[3].hist(log_mag[np.isfinite(log_mag)], bins=100)
ax[3].set_title("Noise floor")
ax[3].set_xlabel("Magnitude (dB re peak)"), ax[3].set_ylabel("Bins")

ring = (radius > 0) & (radius <= nyquist)
ax[4].hist(theta[ring], bins=36, range=(-np.pi, np.pi), weights=power_2d[ring])
ax[4].set_title("Angular power")
ax[4].set_xlabel("Angle (rad)"), ax[4].set_ylabel("Power")

for a in ax: a.grid(alpha=0.25)

---
# STATISTICS
How often each brightness occurs, and how alike nearby pixels are: the predictability (redundancy) an efficient code should remove.

- Input is `log(L + dark)`, the photoreceptor's Weber compression; dark light = RMS render noise of the mean (Barlow), finite at L = 0
- A second render with an independent sampler scramble (`seed=2`) shares the signal only, so WHITENING can split signal from noise
- First order: the shuffled twin has exactly the same histogram but no scene, so pixels are not independent (Attneave 1954)
- Second order: pixels `d` column pitches apart along a row, the same true distance on square and hex lattices
- Separations 1, 2 and 4 replicate Simoncelli & Olshausen 2001 fig. 1: correlation falls with distance but stays high, i.e. redundancy
- Correlation vs distance is the normalised row autocovariance, the Fourier pair of the power spectrum summed over fy (Wiener-Khinchin)

In [ ]:
# seed 2 rescrambles the sampler, so this half's noise is independent of frame's and the two share signal only
half = scene.render(camera=rendercam, aovs=("luminance",), seed=2, show_sky=show_sky)["luminance"][..., 0]
# Dark light = RMS noise of the mean: Weber-like log above the noise floor, finite at L = 0
dark = np.sqrt(np.mean(0.25 * (img - half) ** 2))
halves = [np.log(x + dark) for x in (img, half)]
log_luminance = 0.5 * (halves[0] + halves[1])
shuffled = np.random.default_rng(0).permutation(log_luminance.ravel()).reshape(log_luminance.shape)

correlation = lambda x, d: np.corrcoef(x[:, :-d].ravel(), x[:, d:].ravel())[0, 1]
# Up to half the width, so every separation still pairs at least half the pixels
distances = np.arange(1, img.shape[1] // 2 + 1)
falloff = [correlation(log_luminance, d) for d in distances]

In [ ]:
previews = [lattice_preview(x[..., None], rendercam) for x in (log_luminance, shuffled)]
fig, ax = plt.subplots(1, 3, figsize=[24, 5], width_ratios=[*(p.shape[1] / p.shape[0] for p in previews), 1])
for a, p, name in zip(ax, previews, ("Log luminance", "Shuffled")):
    a.imshow(p, cmap="gray")
    a.set_title(name)
    a.set_xticks([]), a.set_yticks([])

ax[2].hist(log_luminance.ravel(), bins="auto", histtype="step")
ax[2].set_title("Histogram (shared)")
ax[2].set_xlabel("log(L + dark)"), ax[2].set_ylabel("Pixels")
ax[2].set_box_aspect(1), ax[2].grid(alpha=0.25)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=[24, 6])
for a, d in zip(ax, (1, 2, 4)):
    a.plot(log_luminance[:, :-d].ravel(), log_luminance[:, d:].ravel(), ",")
    a.set_title(f"Separation {d}, r = {falloff[d - 1]:.3f}")
    a.set_xlabel("x"), a.set_ylabel(f"x + {d}")

ax[3].plot(distances, falloff, lw=1)
ax[3].set_title("Correlation vs distance")
ax[3].set_xlabel("Distance (column pitches)"), ax[3].set_ylabel("Correlation")
ax[3].set_ylim(top=1)

for a in ax: a.set_box_aspect(1), a.grid(alpha=0.25)

---
# WHITENING
A model of the eye's first filter: it removes what neighbouring pixels predict about each other, suppresses noise and leaves edges.

- Filter `K = sqrt(R) / (R + N)` (Dayan & Abbott eq. 4.49): Wiener gain `R / (R + N)` times whitening `R^-1/2`
- `R` and `N` are measured, not modelled, from STATISTICS' two independently sampled log luminance renders
- Isotropic like a ganglion cell's centre-surround, so per ring of radius: cross power of the two halves estimates signal `R`, a quarter of their difference power the mean's noise `N`
- Signal falls as a power law `f^-a`, i.e. scale invariance; natural images give a ≈ 2 (Field 1987; Ruderman 1997); fitted where R > N
- Whitened power is `R / (R + N)`: flat (decorrelated) where SNR is high, falling to zero where render noise dominates
- Receptive field is K's impulse response: the localised (ZCA) whitening basis `U Λ^-1/2 Uᵀ`, centre-surround like LGN on/off cells
- PCA's `Λ^-1/2 Uᵀ` whitens equally but gives global Fourier checkerboards; the noise term sets the surround's extent (Atick & Redlich 1992)
- Residual neighbour correlation is the noise roll-off plus the scene's horizontal/vertical bias, which an isotropic filter cannot flatten
- DC is removed, so the output is zero-mean contrast; what survives is edges, the phase structure second-order statistics cannot explain

In [ ]:
first, other = (fft(x, twiddle) for x in halves)

ac = radius > 0
# Clamped so the top edge closes the last ring, as in np.histogram
ring_index = np.minimum(np.digitize(radius, edges) - 1, centres.size - 1)
count = np.bincount(ring_index[ac], minlength=centres.size)
ring_mean = lambda v: np.divide(np.bincount(ring_index[ac], weights=v[ac], minlength=centres.size), count, out=np.full(centres.size, np.nan), where=count > 0)

signal = np.maximum(ring_mean(np.real(first * other.conj())), 0.0)
noise = ring_mean(np.abs(first - other) ** 2 / 4)
gain = np.sqrt(signal) / (signal + noise)
log_spectrum = 0.5 * (first + other)
transfer = np.where(ac, gain[ring_index], 0.0)
whitened_spectrum = log_spectrum * transfer
whitened = ifft(whitened_spectrum, twiddle).real
receptive_field = ifft(transfer, twiddle).real
# One wavelength of the gain peak, where R = N; an even row roll keeps each row's hex offset parity
wavelength = 1 / centres[inside][np.argmax(gain[inside])]
rows, cols = 2 * int(np.ceil(wavelength / (2 * row_pitch))), int(np.ceil(wavelength))
field = np.roll(receptive_field, (rows, cols), axis=(0, 1))[:2 * rows + 1, :2 * cols + 1]
whitened_power = ring_mean(np.abs(whitened_spectrum) ** 2)

# Fitted only where signal beats noise: beyond that the R estimate is noise-limited
fit = inside & (signal > noise)
exponent, scale = np.polyfit(np.log(centres[fit]), np.log(signal[fit]), 1)

print(f"Horizontal neighbour correlation: input {correlation(log_luminance, 1):.3f}, whitened {correlation(whitened, 1):.3f}")

In [ ]:
previews = [lattice_preview(x[..., None], rendercam) for x in (whitened, field)]
# Even hex rows sit at x + 1/4 (pixelCentre), a quarter pitch left of mid-frame: trimming row_shift off the right re-centres the field
previews[1] = previews[1][:, :round(previews[1].shape[1] * (1 - row_shift / field.shape[1]))]
# Image columns weighted by their aspect and plots square, so all four panels share one height on either lattice
fig, ax = plt.subplots(1, 4, figsize=[30, 5], width_ratios=[*(p.shape[1] / p.shape[0] for p in previews), 1, 1])

for a, p, name in zip(ax, previews, ("Whitened", "Receptive field")):
    limit = np.abs(p).max()
    a.imshow(p, cmap="gray", vmin=-limit, vmax=limit)
    a.set_title(name)
    a.set_xticks([]), a.set_yticks([])

ax[2].loglog(centres[inside], signal[inside], lw=1, label="Signal R")
ax[2].loglog(centres[inside], noise[inside], lw=1, label="Noise N")
ax[2].loglog(centres[fit], np.exp(scale) * centres[fit] ** exponent, "k--", lw=1, label=f"Power law f^{exponent:.2f}")
ax[2].set_title("Signal and noise")
ax[2].set_xlabel("Radius (cycles/pitch)"), ax[2].set_ylabel("Power")
# Log axes spanning few decades label minor ticks too, which overlap at this panel width
ax[2].tick_params(axis="x", which="minor", labelbottom=False)
ax[2].legend()

ax[3].plot(centres[inside], whitened_power[inside], lw=1)
ax[3].set_title("Whitened power")
ax[3].set_xlabel("Radius (cycles/pitch)"), ax[3].set_ylabel("R / (R + N)")
ax[3].set_ylim(bottom=0)

for a in ax[2:]: a.set_box_aspect(1), a.grid(alpha=0.25)

---
# PHASE
Keeps how strong each wave is but scrambles where it sits: the scene turns to smoke, so edges live in the waves' positions.

- Oppenheim & Lim 1981: edges need the phases of many frequencies aligned at one place; magnitude alone fixes only pairwise correlation
- Each pair shares one power spectrum exactly, so pairwise pixel correlations match; the scrambled twin keeps none of the edges
- Random phase is the angle of the lattice DFT of white Gaussian noise: uniform, and conjugate-symmetric, so the image stays real
- A random-phase twin is the Gaussian image model's view (Simoncelli & Olshausen 2001): right pairwise correlations, nothing more, so smoke
- DC is removed from every image, so all four are zero-mean and compare like for like

In [ ]:
rng = np.random.default_rng(0)
random_phase = np.exp(1j * np.angle(fft(rng.standard_normal(img.shape), twiddle)))
log_ac = np.where(ac, log_spectrum, 0.0)

images = {
    "Log luminance": ifft(log_ac, twiddle).real,
    "Random phase": ifft(np.abs(log_ac) * random_phase, twiddle).real,
    "Whitened": whitened,
    "Whitened, random phase": ifft(np.abs(whitened_spectrum) * random_phase, twiddle).real,
}

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=[20, 10])
for a, (name, x) in zip(ax.ravel(), images.items()):
    limit = np.abs(x).max()
    a.imshow(lattice_preview(x[..., None], rendercam), cmap="gray", vmin=-limit, vmax=limit)
    a.set_title(name)
    a.set_xticks([]), a.set_yticks([])

---
# SPARSITY
Edges make images peaky, mostly near zero with rare large values; scrambled images lose this and become bell-shaped noise.

- Excess kurtosis `m4 / m2^2 - 3` is 0 for a Gaussian and 3 for a Laplace; positive means super-Gaussian, i.e. sparse (Field 1994)
- A random-phase twin sums many waves of independent phase, so the central limit theorem pulls it towards Gaussian
- That pull is weak when power sits in a few low frequencies, so the log luminance twin can stay measurably off zero
- Whitening spreads power over all frequencies: its twin is Gaussian, while the true-phase image keeps the heavy tails of its edges
- Histograms are in standard deviations (every image is zero-mean); the references are the unit-variance Gaussian and Laplace densities

In [ ]:
kurtosis = lambda x: np.mean(x ** 4) / np.var(x) ** 2 - 3
display(pd.Series({name: kurtosis(x) for name, x in images.items()}, name="excess kurtosis").to_frame())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=[16, 8])
for a, pair in zip(ax, (("Log luminance", "Random phase"), ("Whitened", "Whitened, random phase"))):
    standard = [images[name] / images[name].std() for name in pair]
    density = [a.hist(x.ravel(), bins="auto", density=True, histtype="step", log=True, label=name)[0] for name, x in zip(pair, standard)]
    reach = max(np.abs(x).max() for x in standard)
    z = np.linspace(-reach, reach)
    a.plot(z, np.exp(-z ** 2 / 2) / np.sqrt(2 * np.pi), "k--", lw=1, label="Gaussian")
    a.plot(z, np.exp(-np.sqrt(2) * np.abs(z)) / np.sqrt(2), "k:", lw=1, label="Laplace")
    # Floor at one sample in a bin, the lowest density the histograms can show, so the reference tails don't squash the data
    a.set_ylim(bottom=min(d[d > 0].min() for d in density))
    a.set_title(pair[0])
    a.set_xlabel("Standard deviations"), a.set_ylabel("Density")
    a.legend()
    a.set_box_aspect(1), a.grid(alpha=0.25)

---
# PCA
The main directions of colour variation: mostly overall brightness, then colour differences, and where each shows in the image.

- Runs on linear beauty, not the encoded image, so axes reflect radiance rather than display gamma
- PC1 close to (1, 1, 1)/√3 means variation is mostly brightness; PC2 and PC3 then carry chroma
- Projection is each pixel's coordinate along that axis; distance is its distance from that axis (norm over the other two)

In [ ]:
_, variance, axes, projection = principal_components(frame["beauty"])
display(pd.DataFrame(axes.T, index=("PC1", "PC2", "PC3"), columns=("R", "G", "B")).assign(variance=variance, explained=variance / variance.sum()))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=[18, 7])
for i in range(3):
    ax[0, i].imshow(projection[..., i])
    ax[0, i].set_title(f"PC{i + 1} projection")
    distance = np.linalg.norm(np.delete(projection, i, axis=-1), axis=-1)
    ax[1, i].imshow(distance, cmap="gray")
    ax[1, i].set_title(f"PC{i + 1} distance")
for a in ax.flat: a.set_xticks([]), a.set_yticks([])
plt.tight_layout()

---
# ICA
Learns small detectors whose outputs are as independent as possible: on natural images they become oriented edge detectors, like V1 simple cells.

- Patches are WHITENING's receptive field window, a gain-peak wavelength each side of centre, so their size is measured rather than chosen
- Hex rows repeat every two, so patches start on even rows only and all share one geometry; square patches start on every row
- Whitened by patch PCA, not WHITENING's filter: FastICA needs exactly sphered input (Hyvärinen & Oja 2000), and K leaves `R / (R + N)`, not 1
- A component is kept where its signal `λ - n` beats its noise `n`, WHITENING's R > N test per component; `n` is the halves' half-difference
- The rest would be fitted to render noise: this is van Hateren & van der Schaaf's (1998) PCA dimension reduction, with a measured cut
- Symmetric FastICA with `g = tanh`, the contrast `log cosh` (Hyvärinen 1999): robust for the super-Gaussian sources SPARSITY found
- ICA filters come out localised, oriented and band-pass, Gabor-like (Bell & Sejnowski 1997; van Hateren & van der Schaaf 1998)
- PCA `Λ^-1/2 Uᵀ`, ZCA `U Λ^-1/2 Uᵀ` and ICA `W Λ^-1/2 Uᵀ` whiten equally; ICA's rotation `W` alone seeks sparse responses, so its kurtosis is highest
- The window spans a few pitches at the default 64x32, so filters are pixel-scale edges and corners; larger renders widen it to show gratings

In [ ]:
# Window = WHITENING's receptive field; hex rows repeat every two, so even-row starts give every patch one geometry
row_step = 2 if row_shift else 1
patches = lambda x: np.lib.stride_tricks.sliding_window_view(x, field.shape)[::row_step].reshape(-1, field.size)
_, variance, axes, projection = principal_components(patches(log_luminance))
# Noise of the mean along each axis, from the halves' half-difference as WHITENING's N
noise = np.var(patches(halves[0] - halves[1]) / 2 @ axes, axis=0, ddof=1)
# Signal λ - n beats noise n: WHITENING's R > N per component; the rest ICA would fit to render noise
keep = variance > 2 * noise
sphering = axes[:, keep] / np.sqrt(variance[keep])
z = projection[:, keep] / np.sqrt(variance[keep])

unmixing = independent_components(z, np.random.default_rng(0))
# FastICA's components are unordered: sparsest first
unmixing = unmixing[np.argsort([kurtosis(y) for y in (z @ unmixing.T).T])[::-1]]
# Each code is z rotated: PCA not at all, ZCA back onto the pixel axes, ICA by W; its filters are that rotation mapped to pixels
rotations = {"PCA": np.eye(z.shape[1]), "ZCA": axes[:, keep].T, "ICA": unmixing.T}
filters = {name: (sphering @ r).T for name, r in rotations.items()}
kurtoses = {name: np.array([kurtosis(y) for y in (z @ r).T]) for name, r in rotations.items()}

print(f"{len(z)} patches of {field.shape[0]}x{field.shape[1]}: {keep.sum()} of {keep.size} components above the noise")
display(pd.DataFrame({name: pd.Series(k) for name, k in kurtoses.items()}).agg(["median", "mean"]).T.rename_axis("excess kurtosis"))

In [ ]:
def mosaic(f):
    # Each filter at ±1 with a one-cell gap; the tile height field.shape[0] + 1 is even, so every hex tile starts on an even row
    tile = (field.shape[0] + 1, field.shape[1] + 1)
    # Near-square in pitches: tile rows are row_pitch tall
    columns = int(np.ceil(np.sqrt(len(f) * tile[0] * row_pitch / tile[1])))
    rows = -(-len(f) // columns)
    tiles = np.pad((f / np.abs(f).max(axis=1, keepdims=True)).reshape(-1, *field.shape), ((0, rows * columns - len(f)), (0, 1), (0, 1)))
    return lattice_preview(tiles.reshape(rows, columns, *tile).transpose(0, 2, 1, 3).reshape(rows * tile[0], columns * tile[1], 1), rendercam)

previews = {name: mosaic(f) for name, f in filters.items()}
fig, ax = plt.subplots(1, 4, figsize=[30, 8], width_ratios=[*(p.shape[1] / p.shape[0] for p in previews.values()), 1])
for a, (name, p) in zip(ax, previews.items()):
    a.imshow(p, cmap="gray", vmin=-1, vmax=1)
    a.set_title(f"{name} filters")
    a.set_xticks([]), a.set_yticks([])

for name, k in kurtoses.items():
    ax[3].plot(np.linspace(0, 1, k.size), np.sort(k)[::-1], lw=1, label=name)
ax[3].set_title("Response kurtosis")
ax[3].set_xlabel("Rank (fraction of components)"), ax[3].set_ylabel("Excess kurtosis")
ax[3].legend()
ax[3].set_box_aspect(1), ax[3].grid(alpha=0.25)

---
# SPARSE CODING
Rebuilds each patch from as few parts as possible, out of a dictionary with twice as many parts as dimensions: it learns ICA's kind of edges.

- Generative model `z = Φa + ε` (Olshausen & Field 1996, 1997): Gaussian noise of variance σ² and a Laplace prior of scale b on each coefficient
- σ² is measured, the mean noise fraction of ICA's kept components; b matches the prior's energy `K · 2b²` to the signal's `d(1 - σ²)`
- MAP coefficients solve the lasso `½‖z - Φa‖² + (σ²/b)‖a‖₁` (Tibshirani 1996) by FISTA (Beck & Teboulle 2009): most are exactly zero
- Each atom is refitted exactly with the others fixed, then projected onto the unit ball (Mairal et al. 2010), so there is no learning rate
- Both loops stop once the log posterior gains under 1 nat, a posterior ratio below e: "barely worth mentioning" (Jeffreys 1961; Kass & Raftery 1995)
- K = 2d is the smallest integer overcompleteness, which keeps several patches per atom at the default 64x32
- ICA is the noiseless, complete limit: one filter per dimension, inverted exactly; sparse coding infers a few atoms per patch, the rest silent
- Both live in whitened space: each ICA basis vector (a row of W) is matched by |cos| to its nearest atom, and each atom to its nearest ICA vector
- Atoms with no close ICA vector are what overcompleteness adds; a residual below σ² shows the extra atoms also absorb some render noise

In [ ]:
# σ² per whitened dimension: the noise fraction of each kept component, averaged
noise_fraction = np.mean(noise[keep] / variance[keep])
dictionary, codes = sparse_coding(z, 2 * z.shape[1], noise_fraction, np.random.default_rng(0))
# Most used first, by mean |a|, the Laplace prior's own measure
order = np.argsort(np.abs(codes).mean(axis=0))[::-1]
dictionary, codes = dictionary[:, order], codes[:, order]
# W's rows are unit-norm; atoms lie on or inside the unit ball, so normalise them for |cos|
overlap = np.abs(unmixing @ (dictionary / np.linalg.norm(dictionary, axis=0)))
residual = np.mean((z - codes @ dictionary.T) ** 2)

codes_by_model = {"ICA": z @ unmixing.T, "Sparse coding": codes}
print(f"{dictionary.shape[1]} atoms for {z.shape[1]} dimensions; residual {residual:.3f} per dimension vs noise {noise_fraction:.3f}")
display(pd.DataFrame({
    "median excess kurtosis": [np.median([kurtosis(y) for y in c.T]) for c in codes_by_model.values()],
    "exact zeros": [np.mean(c == 0) for c in codes_by_model.values()],
}, index=list(codes_by_model)))

In [ ]:
# Basis functions in pixels: whitened vectors mapped back through U Λ^1/2
unwhitening = axes[:, keep] * np.sqrt(variance[keep])
previews = {"ICA basis": mosaic((unwhitening @ unmixing.T).T), "Sparse coding atoms": mosaic((unwhitening @ dictionary).T)}
fig, ax = plt.subplots(1, 3, figsize=[30, 8], width_ratios=[*(p.shape[1] / p.shape[0] for p in previews.values()), 1])
for a, (name, p) in zip(ax, previews.items()):
    a.imshow(p, cmap="gray", vmin=-1, vmax=1)
    a.set_title(name)
    a.set_xticks([]), a.set_yticks([])

ax[2].hist(overlap.max(axis=1), bins="auto", range=(0, 1), histtype="step", label="ICA vector to nearest atom")
ax[2].hist(overlap.max(axis=0), bins="auto", range=(0, 1), histtype="step", label="Atom to nearest ICA vector")
ax[2].set_title("Best match")
ax[2].set_xlabel("|cos|"), ax[2].set_ylabel("Vectors")
ax[2].legend()
ax[2].set_box_aspect(1), ax[2].grid(alpha=0.25)